# Sampling Procatice - Time-Wise Mismatch + Ensemble-Ready Sampling

This notebook shows practical coding patterns to get the **correct sample** when datasets are time-misaligned or imbalanced, and to prepare samples for ensemble models.

Focus areas:
1. Detect time mismatch and leakage risk
2. Align time-series features safely
3. Build time-aware train/validation/test splits
4. Create balanced temporal samples
5. Generate block-bootstrap samples for bagging/ensembles


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1) Setup and Example Data Loading

Update file paths to your real datasets.


In [ ]:
PROJECT_ROOT = Path('..').resolve()
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'yellow_taxi_24months_complete.xlsx'
ZONE_LOOKUP_PATH = PROJECT_ROOT / 'data' / 'external' / 'taxi_zone_lookup.csv'

print('Project root:', PROJECT_ROOT)
print('Raw exists:', RAW_PATH.exists())
print('Zone lookup exists:', ZONE_LOOKUP_PATH.exists())


## 2) Time Mismatch Diagnostics


In [ ]:
def diagnose_time_coverage(df: pd.DataFrame, ts_col: str, freq: str = 'h') -> pd.DataFrame:
    tmp = df.copy()
    tmp[ts_col] = pd.to_datetime(tmp[ts_col], errors='coerce')
    tmp = tmp.dropna(subset=[ts_col]).sort_values(ts_col)

    full_idx = pd.date_range(tmp[ts_col].min().floor(freq), tmp[ts_col].max().ceil(freq), freq=freq)
    observed = tmp[ts_col].dt.floor(freq).value_counts().sort_index()
    cov = pd.DataFrame(index=full_idx)
    cov['observed_count'] = observed.reindex(full_idx).fillna(0).astype(int)
    cov['is_missing_slot'] = (cov['observed_count'] == 0).astype(int)
    return cov


def find_timestamp_gaps(df: pd.DataFrame, ts_col: str, expected_step='1H') -> pd.DataFrame:
    x = pd.to_datetime(df[ts_col], errors='coerce').dropna().sort_values().drop_duplicates()
    delta = x.diff()
    expected = pd.Timedelta(expected_step)
    gaps = pd.DataFrame({'timestamp': x, 'delta': delta})
    return gaps[gaps['delta'] > expected]


In [ ]:
# Demo synthetic time mismatch dataset
rng = pd.date_range('2025-01-01', periods=200, freq='h')
base = pd.DataFrame({'pickup_hour': rng, 'value': np.random.poisson(20, size=len(rng))})

# Drop random time windows to simulate mismatch
mask = np.ones(len(base), dtype=bool)
mask[20:26] = False
mask[100:108] = False
mis = base[mask].copy()

coverage = diagnose_time_coverage(mis, 'pickup_hour', freq='h')
gaps = find_timestamp_gaps(mis, 'pickup_hour', expected_step='1H')

print('Missing slots:', int(coverage['is_missing_slot'].sum()))
print('Detected long gaps:', len(gaps))
coverage.head()


## 3) Safe Time Alignment (Avoid Leakage)

Use `merge_asof` with backward direction so each target row only sees known past information.


In [ ]:
def safe_time_align(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_ts: str,
    right_ts: str,
    by_cols: list[str] | None = None,
    tolerance='2H',
) -> pd.DataFrame:
    l = left_df.copy().sort_values(left_ts)
    r = right_df.copy().sort_values(right_ts)

    l[left_ts] = pd.to_datetime(l[left_ts], errors='coerce')
    r[right_ts] = pd.to_datetime(r[right_ts], errors='coerce')

    out = pd.merge_asof(
        l,
        r,
        left_on=left_ts,
        right_on=right_ts,
        by=by_cols,
        direction='backward',
        tolerance=pd.Timedelta(tolerance),
    )
    return out


In [ ]:
# Demo: target demand table + delayed weather table
zones = [132, 138, 161]
demand = pd.DataFrame({
    'pickup_hour': np.tile(pd.date_range('2025-01-01', periods=72, freq='h'), len(zones)),
    'zone_id': np.repeat(zones, 72),
    'demand': np.random.poisson(25, size=72*len(zones))
})
weather = pd.DataFrame({
    'wx_time': pd.date_range('2025-01-01', periods=50, freq='2h'),
    'temp_c': np.random.normal(6, 3, size=50),
    'precip': np.random.binomial(1, 0.2, size=50)
})

aligned = safe_time_align(demand, weather, 'pickup_hour', 'wx_time', by_cols=None, tolerance='3H')
aligned[['pickup_hour','zone_id','demand','temp_c','precip']].head()


## 4) Time-Aware Splits (No Future Leakage)


In [ ]:
def rolling_time_splits(
    df: pd.DataFrame,
    ts_col: str,
    train_hours: int,
    valid_hours: int,
    step_hours: int,
):
    x = df.copy().sort_values(ts_col)
    x[ts_col] = pd.to_datetime(x[ts_col], errors='coerce')
    times = np.array(sorted(x[ts_col].dropna().dt.floor('h').unique()))

    start = 0
    while start + train_hours + valid_hours <= len(times):
        tr_t = set(times[start:start+train_hours])
        va_t = set(times[start+train_hours:start+train_hours+valid_hours])
        tr = x[x[ts_col].dt.floor('h').isin(tr_t)].copy()
        va = x[x[ts_col].dt.floor('h').isin(va_t)].copy()
        yield tr, va
        start += step_hours


In [ ]:
splits = list(rolling_time_splits(aligned, 'pickup_hour', train_hours=36, valid_hours=12, step_hours=12))
print('Number of rolling splits:', len(splits))
print('First split shapes:', splits[0][0].shape, splits[0][1].shape)


## 5) Balanced Temporal Sampling

Useful when specific zones/hours are underrepresented.


In [ ]:
def balanced_time_sampling(
    df: pd.DataFrame,
    group_cols: list[str],
    target_per_group: int,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    out_parts = []

    for _, g in df.groupby(group_cols):
        if len(g) >= target_per_group:
            idx = rng.choice(g.index.to_numpy(), size=target_per_group, replace=False)
        else:
            idx = rng.choice(g.index.to_numpy(), size=target_per_group, replace=True)
        out_parts.append(df.loc[idx])

    return pd.concat(out_parts, ignore_index=True)


In [ ]:
aligned['hour'] = aligned['pickup_hour'].dt.hour
balanced = balanced_time_sampling(aligned.dropna(subset=['demand']), ['zone_id', 'hour'], target_per_group=8)
print('Original rows:', len(aligned), '| Balanced rows:', len(balanced))
balanced[['zone_id','hour']].value_counts().head()


## 6) Block Bootstrap Sampling for Bagging/Ensembles

Instead of random row bootstrap, sample **time blocks** to preserve temporal structure.


In [ ]:
def block_bootstrap_sample(
    df: pd.DataFrame,
    ts_col: str,
    block_hours: int = 24,
    n_blocks: int = 30,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    x = df.copy().sort_values(ts_col)
    x[ts_col] = pd.to_datetime(x[ts_col], errors='coerce')

    unique_hours = np.array(sorted(x[ts_col].dt.floor('h').dropna().unique()))
    if len(unique_hours) <= block_hours:
        return x.copy()

    max_start = len(unique_hours) - block_hours
    starts = rng.integers(0, max_start + 1, size=n_blocks)

    parts = []
    for s in starts:
        block_set = set(unique_hours[s:s+block_hours])
        part = x[x[ts_col].dt.floor('h').isin(block_set)]
        parts.append(part)

    return pd.concat(parts, ignore_index=True)


In [ ]:
boot = block_bootstrap_sample(aligned.dropna(subset=['demand']), 'pickup_hour', block_hours=12, n_blocks=20)
print('Bootstrap sample rows:', len(boot))


## 7) Practical Checklist for Correct Sampling in Your Project

1. Always sort by time before split/sampling.
2. Never shuffle globally across time for forecasting tasks.
3. Use backward as-of joins for external features.
4. Use rolling validation windows (not random CV).
5. For ensemble bagging on time series, use block bootstrap.
6. For sparse zones/segments, use balanced group sampling.
7. Track sample counts by zone/hour before and after sampling.


In [ ]:
# Optional quick visualization: missing-time profile from diagnostics
coverage_plot = coverage.copy()
plt.figure(figsize=(12, 3.8))
plt.plot(coverage_plot.index, coverage_plot['observed_count'], color='#355C7D')
plt.title('Observed Count per Hour (Time Mismatch Diagnostic)')
plt.ylabel('count')
plt.xlabel('time')
plt.grid(color='#D1D5DB', linewidth=0.7)
plt.tight_layout()
plt.show()
